# SINAL LIBRAS — auditoria do dataset UECE

Este notebook usa o ambiente gratuito do Google Colab para baixar e inspecionar o pacote de landmarks (aprox. 774 MB). Não treina modelo, não envia dados pessoais e não modifica o dataset original. O uso do corpus deve respeitar a licença CC BY-NC 4.0 e a atribuição aos autores.

**Atenção:** o download pode demorar e consumir cerca de 774 MB de tráfego e espaço temporário.

In [ ]:
from pathlib import Path
import hashlib, json, requests, subprocess, sys, os, time

RECORD_ID = '21398895'
EXPECTED_MD5 = '415a18d9be6d65468bcf92db5e940df8'
WORK = Path('/content/sinal_libras_uece')
WORK.mkdir(exist_ok=True)
print('Ambiente preparado:', WORK)

In [ ]:
# Consulta a API oficial do Zenodo e localiza exatamente Landmarks.zip
api = requests.get(f'https://zenodo.org/api/records/{RECORD_ID}', timeout=60)
api.raise_for_status()
record = api.json()
matches = [f for f in record['files'] if f['key'].lower() == 'landmarks.zip']
if not matches:
    raise RuntimeError('Landmarks.zip não encontrado no registro; confira a versão do dataset.')
file_info = matches[0]
print('Arquivo:', file_info['key'])
print('Tamanho (MB):', round(file_info['size']/1024**2, 1))
print('Checksum:', file_info['checksum'])
print('Link:', file_info['links']['self'])
if file_info['checksum'].split(':')[-1] != EXPECTED_MD5:
    raise RuntimeError('O checksum publicado mudou. Pare e revise o registro antes de baixar.')

In [ ]:
# Download em blocos, com verificação MD5 ao final
archive = WORK / 'Landmarks.zip'
if not archive.exists() or archive.stat().st_size != file_info['size']:
    h = hashlib.md5()
    total = 0
    with requests.get(file_info['links']['self'], stream=True, timeout=(30, 180)) as response:
        response.raise_for_status()
        with archive.open('wb') as out:
            for chunk in response.iter_content(chunk_size=8*1024*1024):
                if chunk:
                    out.write(chunk); h.update(chunk); total += len(chunk)
                    print(f'Baixados: {total/1024**2:.0f} MB / {file_info["size"]/1024**2:.0f} MB', end='\\r')
    print('\\nMD5 calculado:', h.hexdigest())
else:
    print('Arquivo já existe com tamanho esperado; calculando MD5...')
    h = hashlib.md5()
    with archive.open('rb') as src:
        for chunk in iter(lambda: src.read(8*1024*1024), b''):
            h.update(chunk)
    print('MD5 calculado:', h.hexdigest())
if archive.stat().st_size != file_info['size'] or h.hexdigest() != EXPECTED_MD5:
    raise RuntimeError('Download incompleto ou checksum incorreto; não use esses dados.')
print('Download verificado:', archive)

In [ ]:
# Executa a inspeção sem extrair o ZIP e guarda a saída mesmo se houver erro
import urllib.request
inspector_url = 'https://raw.githubusercontent.com/danyellallk/SINAL-LIBRAS/main/training/inspect_libras_eqt.py'
inspector_path = WORK / 'inspect_libras_eqt.py'
urllib.request.urlretrieve(inspector_url, inspector_path)
result = subprocess.run([sys.executable, str(inspector_path), str(archive), '--limit', '40'], text=True, capture_output=True)
print(result.stdout)
if result.stderr: print('AVISOS/ERROS:', result.stderr)
report = '\n'.join([
    'SINAL LIBRAS — relatório de auditoria UECE',
    f'Registro Zenodo: {RECORD_ID}',
    f'Arquivo: {file_info["key"]}',
    f'Tamanho: {file_info["size"]} bytes',
    f'MD5 esperado/verificado: {EXPECTED_MD5}',
    f'Código de saída do inspetor: {result.returncode}',
    '', 'SAÍDA DO INSPETOR:', result.stdout,
    '', 'STDERR:', result.stderr or '(vazio)',
])
report_path = WORK / 'RELATORIO_AUDITORIA_UECE.txt'
report_path.write_text(report, encoding='utf-8')
print('\nRelatório salvo em:', report_path)
if result.returncode != 0: print('A inspeção encontrou erro. Mesmo assim, baixe e envie o relatório para diagnóstico.')


## Próxima etapa
A auditoria detalhada agora salva um JSON completo e mostra um resumo curto, para evitar que a resposta seja cortada ao compartilhar. Execute o notebook inteiro em **Ambiente de execução → Executar tudo**. Ao terminar, a última célula baixa o relatório TXT e o JSON completo. Envie primeiro o resumo que aparece na célula de auditoria; se necessário, consultaremos o JSON. **Não execute treinamento ainda e não habilite `allow_pickle=True` em arquivos desconhecidos.**


In [ ]:
# Auditoria detalhada: salva o JSON completo e mostra um resumo curto para compartilhar
deep_url = 'https://raw.githubusercontent.com/danyellallk/SINAL-LIBRAS/main/training/audit_libras_eqt_npy.py'
deep_script = WORK / 'audit_libras_eqt_npy.py'
full_json_path = WORK / 'AUDITORIA_DETALHADA_COMPLETA.json'
urllib.request.urlretrieve(deep_url, deep_script)
deep = subprocess.run([sys.executable, str(deep_script), str(archive), '--compact', '--output-json', str(full_json_path)], text=True, capture_output=True)
print(deep.stdout)
if deep.stderr: print('STDERR DA AUDITORIA DETALHADA:', deep.stderr)
with report_path.open('a', encoding='utf-8') as out:
    out.write('\n\nRESUMO DA AUDITORIA DETALHADA:\n')
    out.write(deep.stdout)
    out.write('\nSTDERR DA AUDITORIA DETALHADA:\n' + (deep.stderr or '(vazio)'))
print('Código de saída da auditoria detalhada:', deep.returncode)
print('JSON completo salvo em:', full_json_path)
if deep.returncode != 0: print('A auditoria detalhada falhou; o relatório anterior continua salvo.')


In [ ]:
# Baixa os dois relatórios para o celular Android
from google.colab import files
files.download(str(report_path))
files.download(str(full_json_path))
